In [15]:
import arff
import pandas as pd

# загружаем данные
df = pd.read_csv("c:/Personal/Science/Data/Binary_classification/gallstone.csv")


In [17]:
pd.set_option("display.max_rows", None)
pd.set_option('display.max_columns', None)
df.head()

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,Body Mass Index (BMI),Total Body Water (TBW),Extracellular Water (ECW),Intracellular Water (ICW),Extracellular Fluid/Total Body Water (ECF/TBW),Total Body Fat Ratio (TBFR) (%),Lean Mass (LM) (%),Body Protein Content (Protein) (%),Visceral Fat Rating (VFR),Bone Mass (BM),Muscle Mass (MM),Obesity (%),Total Fat Content (TFC),Visceral Fat Area (VFA),Visceral Muscle Area (VMA) (Kg),Hepatic Fat Accumulation (HFA),Glucose,Total Cholesterol (TC),Low Density Lipoprotein (LDL),High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
0,0,50,0,0,0,0,0,0,185,92.8,27.1,52.9,21.2,31.7,40.0,19.2,80.84,18.88,9,3.7,71.4,23.4,17.8,10.6,39.7,0,102.0,250.0,175.0,40.0,134.0,20.0,22.0,87.0,0.82,112.47,0.0,16.0,33.0
1,0,47,0,1,0,0,0,0,176,94.5,30.5,43.1,19.5,23.6,45.0,32.8,67.20,16.68,15,3.2,60.3,38.8,31.0,18.4,32.7,0,94.0,172.0,108.0,43.0,103.0,14.0,13.0,46.0,0.87,107.10,0.0,14.4,25.0
2,0,61,0,0,0,0,0,0,171,91.1,31.2,47.2,20.1,27.1,43.0,27.3,72.67,16.35,15,3.3,62.9,41.7,24.9,16.2,34.0,0,103.0,179.0,124.0,43.0,69.0,18.0,14.0,66.0,1.25,65.51,0.0,16.2,30.2
3,0,41,0,0,0,0,0,0,168,67.7,24.0,41.4,17.0,24.4,41.0,15.8,84.19,16.90,6,2.9,54.1,9.0,10.7,6.5,29.2,1,69.0,173.0,73.0,59.0,53.0,20.0,12.0,34.0,1.02,94.10,0.0,15.4,35.4
4,0,42,0,0,0,0,0,0,178,89.6,28.3,51.4,20.0,31.4,39.0,20.0,80.02,16.81,8,3.5,68.2,28.6,17.9,10.4,37.4,2,109.0,205.0,154.0,30.0,326.0,27.0,54.0,71.0,0.82,112.47,0.0,16.8,40.6


In [19]:
print(df.dtypes)

Gallstone Status                                    int64
Age                                                 int64
Gender                                              int64
Comorbidity                                         int64
Coronary Artery Disease (CAD)                       int64
Hypothyroidism                                      int64
Hyperlipidemia                                      int64
Diabetes Mellitus (DM)                              int64
Height                                              int64
Weight                                            float64
Body Mass Index (BMI)                             float64
Total Body Water (TBW)                            float64
Extracellular Water (ECW)                         float64
Intracellular Water (ICW)                         float64
Extracellular Fluid/Total Body Water (ECF/TBW)    float64
Total Body Fat Ratio (TBFR) (%)                   float64
Lean Mass (LM) (%)                                float64
Body Protein C

In [21]:
df['Gender'] = df['Gender'].astype(str)
df['Comorbidity'] = df['Comorbidity'].astype(str)
df['Hepatic Fat Accumulation (HFA)'] = df['Hepatic Fat Accumulation (HFA)'].astype(str)
print(df.dtypes)

Gallstone Status                                    int64
Age                                                 int64
Gender                                             object
Comorbidity                                        object
Coronary Artery Disease (CAD)                       int64
Hypothyroidism                                      int64
Hyperlipidemia                                      int64
Diabetes Mellitus (DM)                              int64
Height                                              int64
Weight                                            float64
Body Mass Index (BMI)                             float64
Total Body Water (TBW)                            float64
Extracellular Water (ECW)                         float64
Intracellular Water (ICW)                         float64
Extracellular Fluid/Total Body Water (ECF/TBW)    float64
Total Body Fat Ratio (TBFR) (%)                   float64
Lean Mass (LM) (%)                                float64
Body Protein C

In [23]:
from typing import List, Any
import numpy as np

def clean_dataset(
    df: pd.DataFrame,
    cols_to_drop: List[Any] = None,
    missing_values: List[Any] = None
) -> pd.DataFrame:
    """
    Предобработка датафрейма с учетом пользовательских меток пропусков:
    1. Заполняет пропущенные значения
       - Модой для категориальных/текстовых признаков.
       - Средним значением для числовых признаков.
    2. Удаляет признаки, у которых одно значение занимает 100% строк.
    """
    df_cleaned = df.copy()
    n_rows = len(df_cleaned)
    
    if n_rows == 0:
        return df_cleaned

    # Задаем список меток для пропущенных значений по умолчанию, если не передан свой
    if missing_values is None:
        missing_values = ['', ' ', 'N/A', 'NA', 'null', 'nan', 'NaN', 'None', 'none', '?', '-']

    # 1. Приведение пользовательских пропусков к np.nan и их импутация
    for col in df_cleaned.columns:
        # Заменяем все кастомные метки пропусков на np.nan
        df_cleaned[col] = df_cleaned[col].replace(missing_values, np.nan)

        if df_cleaned[col].isna().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_cleaned[col]):
                # а) Числовые переменные -> средним значением
                mean_val = df_cleaned[col].mean()
                df_cleaned[col] = df_cleaned[col].fillna(mean_val)
            else:
                # б) Категориальные переменные -> модой
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(mode_val[0])

    # 2. Проверка условий для удаления колонок
    for col in df_cleaned.columns:
        series = df_cleaned[col]
        max_value_count = series.value_counts().iloc[0]
        max_value_ratio = max_value_count / n_rows

        # Если одинаковых значений = 100% -> к удалению
        if max_value_ratio == 1:
            cols_to_drop.append(col)

    # Удаляем выявленные колонки
    if cols_to_drop:
        df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    return df_cleaned

df = clean_dataset(df)

In [25]:
y = df["Gallstone Status"]
X = df.drop(["Gallstone Status"], axis=1)

In [27]:
import time
import category_encoders as ce
from skrub import SimilarityEncoder, MinHashEncoder
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from k_fold_encoder import KFoldTargetEncoder
from k_fold_glmm_encoder import KFoldGLMMEncoder

cat_cols = ['Gender', 'Comorbidity', 'Hepatic Fat Accumulation (HFA)']

encoders = {
    "MinHash": lambda: ColumnTransformer([(
                f"minhash_{col}",
                MinHashEncoder(),
                [col]
            )
            for col in cat_cols
        ],
        remainder="passthrough"
    ),
    "Similarity": lambda: ColumnTransformer([
        ("cat", SimilarityEncoder(), cat_cols)], 
        remainder="passthrough"),
   'Leave-one-out': lambda: ce.LeaveOneOutEncoder(cols=cat_cols),
   'K-fold':        lambda: KFoldTargetEncoder(cols=cat_cols),
   "CatBoost":      lambda: ce.CatBoostEncoder(cols=cat_cols),
   "CV-GLMM":       lambda: KFoldGLMMEncoder(cols=cat_cols),
   "WOE":           lambda: ce.WOEEncoder(cols=cat_cols),
   "JS":            lambda: ce.JamesSteinEncoder(cols=cat_cols),
   "MT":            lambda: ce.TargetEncoder(cols=cat_cols),
   'GLMM':          lambda: ce.GLMMEncoder(cols=cat_cols),
   'M-Estimate':    lambda: ce.MEstimateEncoder(cols=cat_cols),
   "OneHot":  lambda: ce.OneHotEncoder(cols=cat_cols),
   "Gray":    lambda: ce.GrayEncoder(cols=cat_cols),
   "BaseN":   lambda: ce.BaseNEncoder(cols=cat_cols),
   "Hashing": lambda: ce.HashingEncoder(cols=cat_cols,n_components=32),
   "Helmert": lambda: ce.HelmertEncoder(cols=cat_cols),
   "Sum":     lambda: ce.SumEncoder(cols=cat_cols),
   "BackwardDifference": lambda: ce.BackwardDifferenceEncoder(cols=cat_cols)
}

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():
    for encoder_name, encoder in encoders.items():
        
        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            enc_instance = encoder()  # Вызываем lambda, получая объект кодировщика
            X_train_encoded = enc_instance.fit_transform(X_train, y_train)
            X_test_encoded = enc_instance.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Empirical",
                    "dataset": "Gallstone",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Empirical, Gallstone, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_gallstone_empirical.csv", index=True)
print("Done!")

Empirical, Gallstone, LogReg, MinHash, 1, 1, 0.8181818181818182, 0.8515834430444547, 0.765625, 0.7457627118644068, 0.7857142857142857, 0.7096774193548387, 0.365842342376709
Empirical, Gallstone, LogReg, MinHash, 1, 2, 0.857421875, 0.8774104295371618, 0.78125, 0.75, 0.875, 0.65625, 0.05017375946044922
Empirical, Gallstone, LogReg, MinHash, 1, 3, 0.857421875, 0.8735213956442613, 0.78125, 0.7741935483870968, 0.8, 0.75, 0.04501032829284668
Empirical, Gallstone, LogReg, MinHash, 1, 4, 0.859375, 0.8125958727220722, 0.75, 0.7333333333333333, 0.7857142857142857, 0.6875, 0.04807686805725098
Empirical, Gallstone, LogReg, MinHash, 1, 5, 0.9546370967741935, 0.9570574488294404, 0.8888888888888888, 0.8852459016393442, 0.9, 0.8709677419354839, 0.05101132392883301
Empirical, Gallstone, LogReg, MinHash, 2, 1, 0.898338220918866, 0.900601143241406, 0.796875, 0.7719298245614035, 0.8461538461538461, 0.7096774193548387, 0.055385589599609375
Empirical, Gallstone, LogReg, MinHash, 2, 2, 0.84375, 0.83036343270

C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, Gallstone, CART, Leave-one-out, 5, 5, 0.5030241935483871, 0.49359959037378387, 0.5079365079365079, 0.27906976744186046, 0.5, 0.1935483870967742, 0.053642988204956055
Empirical, Gallstone, CART, K-fold, 1, 1, 0.6700879765395895, 0.6033986175115207, 0.671875, 0.6440677966101694, 0.6785714285714286, 0.6129032258064516, 0.06115984916687012
Empirical, Gallstone, CART, K-fold, 1, 2, 0.703125, 0.6441532258064516, 0.703125, 0.6984126984126984, 0.7096774193548387, 0.6875, 0.08085918426513672
Empirical, Gallstone, CART, K-fold, 1, 3, 0.625, 0.578125, 0.625, 0.625, 0.625, 0.625, 0.07754206657409668
Empirical, Gallstone, CART, K-fold, 1, 4, 0.609375, 0.5700000000000001, 0.609375, 0.5614035087719298, 0.64, 0.5, 0.06700491905212402
Empirical, Gallstone, CART, K-fold, 1, 5, 0.6512096774193548, 0.5898152027184285, 0.6507936507936508, 0.65625, 0.6363636363636364, 0.6774193548387096, 0.07411551475524902
Empirical, Gallstone, CART, K-fold, 2, 1, 0.7194525904203324, 0.6421065493646139, 0.71875,

C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Empirical, Gallstone, LightGBM, MinHash, 1, 1, 0.8699902248289345, 0.9126975142451899, 0.8125, 0.8125, 0.7878787878787878, 0.8387096774193549, 0.1833178997039795
Empirical, Gallstone, LightGBM, MinHash, 1, 2, 0.8525390625, 0.883704001282779, 0.765625, 0.7457627118644068, 0.8148148148148148, 0.6875, 0.1187441349029541


C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


... (37 outputs left out)

C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have v

Empirical, Gallstone, LightGBM, Similarity, 5, 4, 0.8369140625, 0.8818568928152085, 0.765625, 0.7761194029850746, 0.7428571428571429, 0.8125, 0.1114954948425293
Empirical, Gallstone, LightGBM, Similarity, 5, 5, 0.815524193548387, 0.8436908680523785, 0.7619047619047619, 0.7540983606557377, 0.7666666666666667, 0.7419354838709677, 0.09887051582336426
Empirical, Gallstone, LightGBM, Leave-one-out, 1, 1, 0.6124144672531769, 0.6376879108493041, 0.6875, 0.6666666666666666, 0.6896551724137931, 0.6451612903225806, 0.10994529724121094
Empirical, Gallstone, LightGBM, Leave-one-out, 1, 2, 0.4990234375, 0.5625965828870376, 0.484375, 0.49230769230769234, 0.48484848484848486, 0.5, 0.10066103935241699
Empirical, Gallstone, LightGBM, Leave-one-out, 1, 3, 0.494140625, 0.5313984079636312, 0.546875, 0.5245901639344263, 0.5517241379310345, 0.5, 0.11489224433898926
Empirical, Gallstone, LightGBM, Leave-one-out, 1, 4, 0.615234375, 0.5833834610808482, 0.65625, 0.6666666666666666, 0.6470588235294118, 0.6875, 0

C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, Gallstone, LightGBM, Leave-one-out, 5, 3, 0.42333984375, 0.5124944326199772, 0.5, 0.0, 0.0, 0.0, 0.11796116828918457
Empirical, Gallstone, LightGBM, Leave-one-out, 5, 4, 0.62939453125, 0.6677237537647199, 0.53125, 0.5161290322580645, 0.5333333333333333, 0.5, 0.10880112648010254
Empirical, Gallstone, LightGBM, Leave-one-out, 5, 5, 0.685483870967742, 0.6138582402071028, 0.5238095238095238, 0.2857142857142857, 0.5454545454545454, 0.1935483870967742, 0.12254166603088379
Empirical, Gallstone, LightGBM, K-fold, 1, 1, 0.869012707722385, 0.9156972186934155, 0.8125, 0.8064516129032258, 0.8064516129032258, 0.8064516129032258, 0.14298129081726074
Empirical, Gallstone, LightGBM, K-fold, 1, 2, 0.826171875, 0.8720287592737206, 0.78125, 0.7666666666666667, 0.8214285714285714, 0.71875, 0.12988066673278809
Empirical, Gallstone, LightGBM, K-fold, 1, 3, 0.869140625, 0.9009510828078511, 0.78125, 0.7878787878787878, 0.7647058823529411, 0.8125, 0.13729500770568848
Empirical, Gallstone, LightGBM, 